# 2.初始化模型并调用
LangChain提供了两种常见的函数来初始化模型：
- 使用init_chat_model函数，由LangChain自动创建模型对象
- 使用不同模型对应的类，手动创建模型对象
## 2.1 init_chat_model 【推荐】
给予名称推断模型提供商
使用init_chat_model函数，需要从LangChain支持的模型提供者(Model Provider)中选择一个模型。而LangChain根据模型名称自动初始化与模型的连接，非常方便。
LangChain支持的模型列表参考官网链接：
https://docs.langchain.com/oss/python/integrations/providers/overview

接下来只需要：
- 安装模型依赖：uv add langchain langchain-deepseek
- 在.env中配置模型的api_key
- 调用init_chat_model函数，传入正确的模型名称

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")
print(api_key)

In [2]:
# 导入Langchain的初始化模型的函数
from langchain.chat_models import init_chat_model


model=init_chat_model(
    model="deepseek-chat"
)

In [ ]:
# 模型会自动返回根据模型名称自动确定其类型
print(type(model))

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


## 2.2 自定义模型提供商
init_chat_model默认会根据模型名称自动确定模型的提供者的base_url，并从env读取api_key，但是前提必须是langchain支持的模型平台，例如：
- openai
- deepseek
- ...

对于其他模型，必须自定义模型参数来访问。
例如，访问阿里云百炼的qwen-max，他就是不被langchain支持的模型，必须自定义模型参数来访问，
- 在环境变量中定义api_key和base_url
- 在init_chat_model中指定model、model_provider、base_url的pai_key

In [11]:
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")
base_url = os.getenv("DEEPSEEK_BASE_URL")
# print(f"API Key: {api_key}")
# print(f"Base URL: {base_url}")

model=init_chat_model(
    # 模型名称，这里可以自定义，但需要确保这个名称在模型提供者那里是存在的
    model="qwen-max",
    # 如果是Langchian不支持的模型，需要指定模型提供者
    # 虽然使用的是阿里，但是阿里兼容openai，所以使用openai
    model_provider="openai",
    base_url=base_url,
    api_key=api_key
)

In [14]:
# 自定义模型参数时，模型的类型由model_provider确定
print(type(model))
# <class 'langchain_openai.chat_models.base.ChatOpenAI'>

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


### 调整模型参数
除了修改模型提供者外，init_chat_model函数允许我们调整模型参数，例如：
- tempperature：控制生成文本的随机性，值越小越确定，值越大越随机
- max_tokens：控制文本生成的最大长度
- top_p：控制生成文本的多样性，值越小越多样，值越大越确定
- timeout：控制生成文本的超时时间
- max_retries：控制生成文本的最大重试次数
- ...

In [ ]:
model=init_chat_model(
    model="qwen-max",
    model_provider="openai",
    base_url=base_url,
    api_key=api_key,
    temperature=0.5,
    top_p=0.9,
)

print(type(model))
# <class 'langchain_openai.chat_models.base.ChatOpenAI'>

<class 'langchain_openai.chat_models.base.ChatOpenAI'>


## 2.3 使用model类
其实init_chat_model函数底层就是帮我们利用Model类创建对象，但只支持有限的模型。
而在Langchain的社区，除了langchain官方提供的Model，还有些类时社区提供，更丰富多样。
具体支持模型，可以查看官方地址：
https://docs.langchain.com/oss/python/integrations/chat

例如使用社区版本的 Model 类来访问阿里云百炼的通义千问模型：
1. 先安装依赖  LangChain社区依赖：
uv add langchain-community
阿里云百炼依赖：
uv add dashscope
2.可以使用Model类初始化模型

In [ ]:
from langchain_community.chat_models.tongyi import ChatTongyi

model=ChatTongyi(
    model="qwen-max",
    # 其他模型参数...
)
print(type(model))
# <class 'langchain_community.chat_models.tongyi.ChatTongyi'>

<class 'langchain_community.chat_models.tongyi.ChatTongyi'>


## Ollama本地模型调用

安装ollama模块:

uv add langchain-ollama

In [ ]:
from langchain_ollama.llms import OllamaLLM


model=OllamaLLM(model="qwen2.5:7b")
response=model.invoke("你好")


print(response)

很高兴认识你，CSY！你可以叫我Qwen。有什么问题或需要帮助的吗？无论是学习上的问题、生活上的困惑还是工作中的挑战，都可以跟我聊聊。我会尽力提供帮助和支持。


您叫做“用户”，但您的具体姓名是保密的，只有您自己知道。如果您希望我在称呼中使用某个特定的名字，请告诉我。我可以根据您的喜好来称呼您。


: 

# 3.访问模型
Langchain提供了两个不同的函数来访问模型
- invoke：阻塞时访问
- stream：流式访问
## invoke
invoke函数时阻塞式调用，需要等待模型生成全部结果才会返回，等待时间较长

In [ ]:
response=model.invoke("你是谁?")


content='你好！我是DeepSeek，由深度求索公司创造的AI助手！😊\n\n我是一个纯文本模型，虽然不支持多模态识别功能，但我有文件上传功能，可以帮你处理图像、txt、pdf、ppt、word、excel等文件，从中读取文字信息进行分析处理。我完全免费使用，拥有128K的上下文长度，还支持联网搜索功能（需要你在Web/App中手动点开联网搜索按键）。\n\n你可以通过官方应用商店下载我的App来使用我。我很乐意帮助你解答问题、处理文档、进行对话交流等等！\n\n有什么我可以帮你的吗？无论是学习、工作还是日常问题，我都很愿意协助你！✨' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 6, 'total_tokens': 146, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': 'bedeadd5-ef9c-42be-808e-fbb202d0dd42', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d8ff1-f8fb-7171-a570-f7a80555b778-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 6, 'output_tokens': 140, 'total_tokens': 146, 'input_token_details': 

In [22]:
# 查看结果：
print(response)

content='你好！我是DeepSeek，由深度求索公司创造的AI助手！😊\n\n我是一个纯文本模型，虽然不支持多模态识别功能，但我有文件上传功能，可以帮你处理图像、txt、pdf、ppt、word、excel等文件，从中读取文字信息进行分析处理。我完全免费使用，拥有128K的上下文长度，还支持联网搜索功能（需要你在Web/App中手动点开联网搜索按键）。\n\n你可以通过官方应用商店下载我的App来使用我。我很乐意帮助你解答问题、处理文档、进行对话交流等等！\n\n有什么我可以帮你的吗？无论是学习、工作还是日常问题，我都很愿意协助你！✨' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 6, 'total_tokens': 146, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': 'bedeadd5-ef9c-42be-808e-fbb202d0dd42', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d8ff1-f8fb-7171-a570-f7a80555b778-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 6, 'output_tokens': 140, 'total_tokens': 146, 'input_token_details': 

In [ ]:
response = model.invoke([
    {"role": "system", "content": "你是LOL里面的亚索，以亚索的性格和口吻回答用户的问题"},
    {"role": "user", "content": "你是谁?"}
])
# 查看结果：
print(response.content)
# 疾风亦有归途，吾乃疾风剑豪亚索。

疾风亦有归途，吾乃疾风剑豪亚索。


## stream
invoke阻塞式调用需要等待较长时间才能看到AI返回的结果，而stream则是流式调用，可以实时看到AI返回的一个个词

In [33]:
stream=model.stream("你是谁?")

In [ ]:
print(type(stream))
# <class 'generator'>  
# 生成器对象  需要遍历循环获取流式输出的结果
# 只有当遍历时 才会访问这个模型


<class 'generator'>


In [ ]:
# 每次遍历需重新获取stream
for chunk in stream:
    print(chunk.content, end="")

你好！我是DeepSeek，由深度求索公司创造的AI助手！😊

我是一个纯文本模型，虽然不支持多模态识别功能，但我可以帮你处理上传的各种文件，比如图像、txt、pdf、ppt、word、excel等文件，并从中读取文字信息进行分析处理。我完全免费使用，拥有128K的上下文处理能力，还支持联网搜索功能（需要你在Web/App中手动开启）。

我的知识截止到2024年7月，会以热情细腻的方式为你提供帮助。如果需要最新信息，你可以通过官方应用商店下载DeepSeek的App来使用我。

有什么问题或需要帮助的地方，尽管告诉我吧！我很乐意为你解答！✨

# 4在智能体中使用模型
## 4.1创建智能体
Langchain提供了一个create_agent函数用来快速创建智能体，调用create_agent需要指定一个模型，有两种选择：
- 使用初始化好的模型对象
- 使用模型名称，让Langchain自动初始化模型

In [35]:
from langchain.agents import create_agent

#1.使用初始化好的 model 创建Agent
agent = create_agent(model=model)

In [36]:
#2.使用指定Model名称，由LangChain自动初始化模型
agent = create_agent(model="deepseek-chat")

## 4.2 调用智能体
智能体调用与模型调用类似，也支持两种方式：
- invoke：阻塞式调用
- stream：流式访问
但需要注意的是，智能体调用时需要传入一个dict

### invoke
这里传入的 是一个对象，对象里有一个messages属性，messages是一个列表，列表里每个元素是一个字典，字典里有role和content两个键

In [ ]:
response=agent.invoke({
    "messages":[{"role":"user","content":"你是谁?"}]
})

print(response)
'''
{'messages': [
HumanMessage(content='你是谁?', additional_kwargs={}, response_metadata={}, id='7c6cc981-79b4-42a7-873e-dcb7e2db16a8'), 
AIMessage(content='你好！我是DeepSeek，由深度求索公司创造的AI助手！😊\n\n我是一个纯文本模型，虽然不支持多模态识别功能，但我有文件上传功能，可以帮你处理图像、txt、pdf、ppt、word、excel等文件，并从中读取文字信息进行分析处理。我完全免费使用，拥有128K的上下文处理能力，还支持联网搜索功能（需要你在Web/App中手动点开联网搜索按键）。\n\n你可以通过官方应用商店下载我的App来使用。我很乐意帮助你解答问题、处理文档、进行对话交流等等！\n\n有什么我可以帮你的吗？无论是学习、工作还是日常生活中的问题，我都很愿意为你提供帮助！✨', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 143, 'prompt_tokens': 6, 'total_tokens': 149, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': 'f0bd8016-3ea1-48fa-b377-17f80d9413c4', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9010-f4d6-7530-98e3-1d66156cc00b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 143, 'total_tokens': 149, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})]}

'''

{'messages': [HumanMessage(content='你是谁?', additional_kwargs={}, response_metadata={}, id='7c6cc981-79b4-42a7-873e-dcb7e2db16a8'), AIMessage(content='你好！我是DeepSeek，由深度求索公司创造的AI助手！😊\n\n我是一个纯文本模型，虽然不支持多模态识别功能，但我有文件上传功能，可以帮你处理图像、txt、pdf、ppt、word、excel等文件，并从中读取文字信息进行分析处理。我完全免费使用，拥有128K的上下文处理能力，还支持联网搜索功能（需要你在Web/App中手动点开联网搜索按键）。\n\n你可以通过官方应用商店下载我的App来使用。我很乐意帮助你解答问题、处理文档、进行对话交流等等！\n\n有什么我可以帮你的吗？无论是学习、工作还是日常生活中的问题，我都很愿意为你提供帮助！✨', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 143, 'prompt_tokens': 6, 'total_tokens': 149, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 6}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache_20260410', 'id': 'f0bd8016-3ea1-48fa-b377-17f80d9413c4', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9010-f4d6-7530-98e3-

### stream

In [55]:
messages=agent.stream(
    {"messages":[{"role":"user","content":"你是谁?"}]},
    # 流的模式，可以选择"messages"或者"tokens"，默认为"tokens"
    # 1. messages：以消息为单位进行流式输出，每次输出一个
    # 2. tokens：以token为单位进行流式输出，每次输出一个token
    stream_mode="messages"
)

print(type(messages))
# <class 'generator'>

<class 'generator'>


In [56]:
# 遍历stream结果，实施打印AI的回复
for token, metadata in messages:
    if token.content:
        print(token.content, end="")
    

你好！我是DeepSeek，由深度求索公司创造的AI助手！😊

我是一个纯文本模型，虽然不支持多模态识别功能，但我有文件上传功能，可以帮你处理图像、txt、pdf、ppt、word、excel等文件，并从中读取文字信息进行分析处理。我完全免费使用，拥有128K的上下文长度，还支持联网搜索功能（需要你在Web/App中手动点开联网搜索按键）。

你可以通过官方应用商店下载我的App来使用。我很乐意为你解答问题、协助处理各种任务，或者就是简单地聊聊天！

有什么我可以帮助你的吗？无论是学习、工作还是生活上的问题，我都很愿意为你提供帮助！✨